In [ ]:
# [목적] LLM 답변 캐싱을 비교하기 전에, API 키와 실행 기록 설정을 준비하는 예제입니다.
# 이후 셀에서 같은 질문을 반복 실행할 때 캐시가 응답 시간을 줄이는지 확인할 수 있습니다.

from dotenv import load_dotenv
# LangSmith에 실행 과정과 결과를 기록하는 도구입니다.
from langchain_teddynote import logging

# .env 파일의 API 키를 불러와 모델 호출에 사용합니다.
load_dotenv()
# 이 노트북의 실행 기록을 Chapter7-Models 프로젝트에 모읍니다.
logging.langsmith("Chapter7-Models")

In [ ]:
# [목적] 국가 이름을 넣으면 짧은 요약을 돌려주는 LLM 체인을 만드는 예제입니다.
# 프롬프트의 빈칸에 나라를 넣고 OpenAI 모델에 전달해, 다음 셀에서 답변을 받습니다.

from langchain_core.prompts import PromptTemplate
# OpenAI 채팅 모델에 연결하기 위한 LangChain 클래스입니다.
from langchain_openai import ChatOpenAI

# 사용할 모델을 정해 LLM 연결 객체를 만듭니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# {country}는 실행할 때 실제 국가 이름으로 바뀌는 자리표시자입니다.
prompt = PromptTemplate.from_template("{country}에 대해서 200자 내외로 요약해줘")

# | 연산자로 프롬프트와 모델을 연결해, 입력부터 답변 생성까지 한 번에 처리하는 체인을 만듭니다.
chain = prompt | llm

In [ ]:
# [목적] 캐시를 적용하기 전, 체인을 처음 실행한 응답 시간을 확인하는 예제입니다.
# 한국을 프롬프트의 country 자리에 넣어 모델을 호출하고, 결과를 다음 캐시 실행과 비교합니다.

%%time
# invoke()는 체인에 입력값을 전달하고 완성된 답변을 받을 때 사용합니다.
response = chain.invoke({"country": "한국"})
# 모델 답변의 텍스트 내용만 화면에 출력합니다.
print(response.content)

In [ ]:
# [목적] 메모리 캐시를 켜서, 같은 질문의 답변을 현재 노트북 실행 동안 재사용하는 예제입니다.
# 첫 호출 결과를 메모리에 저장하므로, 다음 셀에서 같은 입력을 실행하면 API 호출을 줄일 수 있습니다.

%%time
# LangChain 전체에서 사용할 캐시를 설정하는 함수입니다.
from langchain_core.globals import set_llm_cache
# 프로그램을 종료하면 사라지는 임시 메모리 캐시입니다.
from langchain_core.caches import InMemoryCache

# 이후의 LLM 호출 결과를 메모리에 저장하도록 설정합니다.
set_llm_cache(InMemoryCache())

# 같은 질문을 한 번 실행해 캐시에 답변을 저장합니다.
response = chain.invoke({"country": "한국"})
# 받은 답변을 화면에 출력합니다.
print(response.content)

In [ ]:
# [목적] 바로 앞 셀에서 저장한 메모리 캐시가 실제로 재사용되는지 확인하는 예제입니다.
# 동일한 입력을 다시 보내 응답 시간이 줄어드는지 확인하며, 결과는 현재 실행 중인 메모리에만 남습니다.

%%time
# 앞 셀과 같은 입력이므로 저장된 답변이 있으면 캐시에서 가져옵니다.
response = chain.invoke({"country": "한국"})
# 캐시에서 가져온 답변도 일반 호출과 같은 방식으로 출력합니다.
print(response.content)

In [ ]:
# [목적] SQLite 파일에 LLM 답변을 저장해, 노트북을 다시 열어도 캐시를 유지하는 예제입니다.
# 메모리 캐시와 달리 답변을 파일로 남기므로, 반복 API 호출 비용과 대기 시간을 줄이는 데 사용합니다.

# SQLite 기반 캐시를 제공하는 별도 LangChain Community 패키지의 클래스입니다.
from langchain_community.cache import SQLiteCache
# LangChain 전체에 사용할 캐시를 설정하는 함수입니다.
from langchain_core.globals import set_llm_cache
import os

# 캐시 데이터베이스를 저장할 폴더가 없을 때만 새로 만듭니다.
if not os.path.exists("cache"):
    os.makedirs("cache")

# 답변을 cache 폴더의 llm_cache.db 파일에 저장하도록 설정합니다.
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))

In [ ]:
# [목적] SQLite 파일 캐시를 적용한 상태에서 체인을 실행해 답변을 저장하거나 재사용하는 예제입니다.
# 같은 질문을 다시 실행하면 파일에 저장된 답변을 사용하므로, 노트북을 재시작한 뒤에도 속도 차이를 확인할 수 있습니다.

%%time

# 한국에 대한 요약을 요청하며, 같은 요청이 저장돼 있으면 SQLite 캐시에서 답변을 가져옵니다.
response = chain.invoke({"country": "한국"})
# 최종 답변 텍스트를 화면에 출력합니다.
print(response.content)